## Transformaciones: Planteamiento

Transformaciones a realizar, orden y sobre que técnica

---
### **1. Objetivo**

Dejar lista, **sobre Spark**, la base relacional y los artefactos que alimentan las siguientes secciones

### **2. Grano de análisis**

- **Una fila = una adjudicación de una entidad a un proveedor en una familia UNSPSC** (entidad–proveedor–familia)

- Las técnicas trabajan sobre **relaciones**, no sobre procesos aislados: 

- La **adjudicación** es la relación efectiva; el contrato falta en el 10.8% de los casos

- **Familia** (4 dígitos UNSPSC) define el mercado: el segmento (2 dígitos) es muy grueso y el producto (8 dígitos) muy disperso
- Todo lo demás (features, matrices, grafos) se agrega desde este grano

---
### 3. Capas de datos

| Capa | Contenido | Grano | Partición |
|---|---|---|---|
| `data/raw/` | JSON OCDS mensual | archivo | año / mes | `data/download.py` |
| `data/bronze/` | OCDS anidado en Parquet | proceso | — |
| `data/staging/` | tablas aplanadas (procesos, adjudicaciones, oferentes, ítems, contratos, partes) | según tabla | `anio` |
| `data/final/` | procesos limpios, hechos, participantes, dimensiones, auditoría de la resolución | entidad–proveedor–familia | `anio/mes` |
| `data/artefactos/` | features, matrices, índices y grafos | según artefacto | — |

---

### **4. Transformaciones**

#### a. Limpieza

- Texto en mayúsculas y espacios colapsados; 1 fila por `ocid`

- Solo procesos con fecha en 2023-2025

- Montos a soles con el tipo de cambio mensual del BCRP; monto ≤ 0 → nulo

- Banderas de competencia: `postor_unico` (n_postores = 1) y `es_directa` (contratación directa)

#### b. Resolución de entidades

| Nivel | Regla | Resultado |
|---|---|---|
| 1. Identificador | El formato del id indica el tipo de actor y define su clave (tabla de abajo) | Determinista |
| 2. Nombre exacto | Mención sin id con el mismo nombre normalizado que **una sola** clave | Determinista |
| 3. MinHash + LSH | Mención sin id → clave más parecida si Jaccard ≥ 0.85 (sin consorcios); si no, `NOM:` | Probabilístico |

**Claves por identificador**

| Así viene en los datos | Qué es | Clave |
|---|---|---|
| `PE-CONSUCODE-200489` | Entidad pública (código del SEACE) | `ENT:200489` |
| `PE-RUC-20100085225` | Empresa (RUC de 11 dígitos, empieza con 20) | `RUC:20100085225` |
| `PE-RUC-10091853928` | Persona natural (RUC 10 = `10` + DNI + dígito) | `PN:09185392` (su DNI) |
| `PE-RUC-1761778` | Consorcio (código de 6-7 dígitos, no es RUC) | `CONS:1761778` |
| `PE-RUC-L0556212336` | Proveedor extranjero (`L` + 10 dígitos) | `EXT:L0556212336` |

#### c. Hechos y participantes

- **Hechos:** adjudicación × proveedor resuelto × familia

- Con `monto_asignado` = monto / ganadores / familias

- **Participantes:** oferentes ∪ ganadores por proceso

---
### **5. Artefactos por técnica**

| Técnica | Transformación | Artefacto |
|---|---|---|
| Clustering y anomalías | Agregar hechos por actor: tamaño, alcance, dependencia (HHI), competencia, tipo | `features_proveedor`, `features_entidad` |
| Recomendación | Pivotear hechos a entidad × proveedor y entidad × familia, por año, con índices enteros | `matriz_entidad_proveedor`, `matriz_entidad_familia`, `indice_*` |
| PageRank y HITS | Aristas dirigidas entidad → proveedor con peso | `grafo_bipartito` |
| Comunidades | Self-join de participantes por proceso → pares de proveedores | `grafo_coparticipacion` |
| Representación | Reutiliza la matriz (SVD) y la co-participación (Node2Vec) | — |

---

### **6. Criterio de aristas**

- **Bipartito:** entidad → proveedor si hubo ≥ 1 adjudicación; peso = n.º de adjudicaciones o monto

- **Co-participación:** proveedor — proveedor si participaron juntos en **≥ 2 procesos** de **2 a 30 participantes**

- **Alternativa:** grafo KNN sobre `features_proveedor` estandarizado, si la co-participación resultara escasa

- **Consorcios:** son un nodo propio; la fuente no publica sus miembros, así que no hay arista consorcio–empresa

### **7. Vínculos incompletos**

| Caso | Tratamiento |
|---|---|
| Adjudicación sin contrato (10.8%) | Se conserva como arista; `tiene_contrato = 0` |
| Contrato sin adjudicación (1.0%) | No genera arista: no tiene proveedor identificable |
| Award sin proveedor (1.9%) | No genera arista |
| Ganador no listado como oferente | Se agrega a participantes (`en_lista_oferentes = 0`, `gano = 1`) |
| Ítem sin UNSPSC | Familia del tender; si no hay, `SIN` (excluida de las matrices de rubro) |

---

### **8. Particionamiento**

- **Disco:** `hechos`, `procesos` y `participantes` por `anio/mes`

- **Self-join de co-participación:** se reparticiona una vez por `ocid` y se cachea

- **Dimensiones chicas** (entidades, mapa de la resolución, tipo de cambio): *broadcast join*, sin mover la tabla grande

- **`spark.sql.shuffle.partitions` = 24:** con 2 workers de 1 core, más particiones solo agregan overhead